# Student Task: Build a RAG Question-Answering System

## Goal
In this task, you will build a Retrieval-Augmented Generation (RAG) pipeline over the provided `RAG and Fine-tuning - Final.pdf` file.

You will implement these stages:

```text
PDF -> Text -> Chunks -> Embeddings -> Retrieval -> Augmented Prompt -> Grounded Answer
```

Complete every section marked `TODO`.

## Learning objectives

By the end of this task, you should be able to:

- Explain why RAG is useful for private, changing, or large information sources.
- Load and prepare a PDF for semantic search.
- Split a document into overlapping chunks.
- Create embeddings and retrieve the most relevant chunks with cosine similarity.
- Augment a prompt with retrieved context.
- Generate an answer that is grounded in the source document.
- Compare RAG with fine-tuning and describe the purpose of LoRA/QLoRA.

## Task requirements

Your final notebook should:

1. Read the supplied PDF with `pypdf`.
2. Create chunks with a configurable size and overlap.
3. Embed all chunks and the user's question with OpenAI.
4. Retrieve the top 3 relevant chunks using cosine similarity.
5. Generate a concise answer using only the retrieved context.
6. Print the retrieved sources, answer, and a simple grounding check.
7. Answer the conceptual questions near the end.

Do not place your API key directly in the notebook.

## 1. Setup

In [ ]:
!pip install -q -U openai pypdf numpy

from getpass import getpass
from pathlib import Path
import re
import numpy as np
from openai import OpenAI
from pypdf import PdfReader

api_key = getpass("Enter your OpenAI API key: ")
client = OpenAI(api_key=api_key)
CHAT_MODEL = "gpt-5-mini"
EMBEDDING_MODEL = "text-embedding-3-small"
PDF_PATH = Path("RAG and Fine-tuning - Final.pdf")

print("Setup complete")

## 2. Load the source document

In [ ]:
# TODO 1: Read every page from PDF_PATH and combine the extracted text.
# Save the result in document_text and print its character count.
document_text = None

print("Document characters:", len(document_text) if document_text else 0)

## 3. Chunk the document

Chunking makes retrieval possible because the system can search meaningful sections instead of sending the entire PDF to the model. Keep the overlap so ideas split across boundaries are less likely to be lost.

In [ ]:
CHUNK_SIZE = 900
CHUNK_OVERLAP = 150

# TODO 2: Write chunk_text(text, chunk_size, overlap).
# Return a list of non-empty overlapping text chunks.
def chunk_text(text, chunk_size=900, overlap=150):
    raise NotImplementedError("Complete this function")

chunks = chunk_text(document_text, CHUNK_SIZE, CHUNK_OVERLAP)
print("Number of chunks:", len(chunks))
print("First chunk preview:\n", chunks[0][:500])

## 4. Create embeddings and an in-memory vector index

The embedding represents meaning as a vector. The index below is intentionally simple: it stores vectors in NumPy and uses cosine similarity.

In [ ]:
# TODO 3: Complete embed_texts so it returns one list of embedding vectors per input text.
def embed_texts(texts):
    response = client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
    return [item.embedding for item in response.data]

# TODO 4: Complete cosine_similarity_matrix.
# It should return the cosine similarity between one query vector and every row in matrix.
def cosine_similarity_matrix(query_vector, matrix):
    raise NotImplementedError("Complete this function")

chunk_embeddings = np.array(embed_texts(chunks), dtype=np.float32)
print("Embedding matrix shape:", chunk_embeddings.shape)

## 5. Retrieve relevant context

In [ ]:
question = "What is the difference between RAG and fine-tuning, and when should each be used?"
TOP_K = 3

# TODO 5: Embed the question, calculate similarities, and select the top TOP_K chunks.
# Save the selected text in retrieved_chunks and their scores in retrieved_scores.
query_embedding = None
retrieved_chunks = []
retrieved_scores = []

for rank, (score, chunk) in enumerate(zip(retrieved_scores, retrieved_chunks), start=1):
    print(f"[{rank}] score={score:.3f}\n{chunk[:400]}\n")

## 6. Augment the prompt and generate a grounded answer

The model must use the retrieved context and must say when the answer is not supported by the source.

In [ ]:
# TODO 6: Build context from retrieved_chunks.
# TODO 7: Write an augmented prompt containing the question and context.
context = None
augmented_prompt = None

# TODO 8: Call the Responses API and save the answer in answer.
answer = None

print("Answer:")
print(answer)

## 7. Evaluate grounding

This is a small educational check, not a complete evaluation system. Inspect whether important terms from the retrieved context appear in the answer and whether the answer admits missing evidence.

In [ ]:
# TODO 9: Normalize text and calculate how many distinctive context terms appear in the answer.
def normalize_words(text):
    return set(re.findall(r"[a-zA-Z]{4,}", text.lower()))

context_terms = normalize_words(context or "")
answer_terms = normalize_words(answer or "")
overlap = context_terms & answer_terms
grounding_ratio = len(overlap) / max(len(answer_terms), 1)

print(f"Distinctive-term overlap: {len(overlap)}")
print(f"Simple grounding ratio: {grounding_ratio:.2%}")
print("Retrieved context used:", bool(context))

## 8. Conceptual reflection

Answer in your own words:

1. What do indexing, chunking, retrieval, and augmentation each do in a RAG workflow?
2. Why can poor retrieval produce a poor answer even when the language model is capable?
3. How is a vector database different from a traditional keyword database?
4. When is RAG a better choice than fine-tuning? When might fine-tuning be better?
5. What do LoRA and QLoRA change during fine-tuning, and why are base model weights often frozen?

## Submission checklist

- [ ] All `TODO` sections are completed.
- [ ] The notebook runs from top to bottom without errors.
- [ ] The PDF is loaded and split into overlapping chunks.
- [ ] The top retrieved chunks and similarity scores are printed.
- [ ] The answer is generated from retrieved context.
- [ ] The grounding check is printed and interpreted.
- [ ] The conceptual reflection questions are answered.
- [ ] No API key is written directly in the notebook.